# Aula: K-Means em Ciência de Dados — agrupando músicas do Spotify

**Disciplina:** ICD-004 Análise de Algoritmos · Fatec Jundiaí
**Professor:** Prof. Humberto Zanetti
**Data:** 24/11/2026

**Integrantes:**
- Wellington Torres
<!-- Novos integrantes: acrescente uma linha "- Nome completo" acima. -->

**Repositório:** [github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify)

> Como usar esta aula: leia de cima para baixo e execute as células em ordem
> (*Ambiente de execução → Executar tudo*). As células **🧪 Experimente** convidam você a mudar
> um parâmetro e rodar de novo.

## Sumário

- [⚙️ Configuração do ambiente](#⚙️-Configuração-do-ambiente)
- [I. Introdução e Contextualização](#I.-Introdução-e-Contextualização)
  - [I.1 Aprendizado não supervisionado](#I.1-Aprendizado-não-supervisionado)
  - [I.2 Clustering (agrupamento)](#I.2-Clustering-(agrupamento))
  - [I.3 Como o K-Means funciona — implementado do zero](#I.3-Como-o-K-Means-funciona-—-implementado-do-zero)
  - [I.4 Quanto custa? Complexidade do K-Means](#I.4-Quanto-custa?-Complexidade-do-K-Means)
  - [I.5 Como avaliar: inércia, Método do Cotovelo e silhueta](#I.5-Como-avaliar:-inércia,-Método-do-Cotovelo-e-silhueta)
- [II. Ferramentas de Trabalho (Scikit-learn e Python)](#II.-Ferramentas-de-Trabalho-(Scikit-learn-e-Python))
  - [II.1 Bibliotecas e importações](#II.1-Bibliotecas-e-importações)
  - [II.2 O KMeans do scikit-learn](#II.2-O-KMeans-do-scikit-learn)
- [III. Escolha do Tema e Preparação dos Dados](#III.-Escolha-do-Tema-e-Preparação-dos-Dados)
  - [III.1 Tema: por que agrupar músicas?](#III.1-Tema:-por-que-agrupar-músicas?)
  - [III.2 O dataset: origem e dicionário de variáveis](#III.2-O-dataset:-origem-e-dicionário-de-variáveis)
  - [III.3 Carregamento e limpeza](#III.3-Carregamento-e-limpeza)
  - [III.4 Escolha das features](#III.4-Escolha-das-features)
  - [III.5 Padronização com StandardScaler](#III.5-Padronização-com-StandardScaler)
- [IV. Implementação e Aplicação Prática](#IV.-Implementação-e-Aplicação-Prática)
  - [IV.1 Definindo k: Método do Cotovelo (e silhueta)](#IV.1-Definindo-k:-Método-do-Cotovelo-(e-silhueta))
  - [IV.2 Treinando o K-Means](#IV.2-Treinando-o-K-Means)
  - [IV.3 Visualizando a separação dos clusters](#IV.3-Visualizando-a-separação-dos-clusters)
  - [IV.4 Centroides em relação aos pontos](#IV.4-Centroides-em-relação-aos-pontos)
- [V. Análise e conclusão](#V.-Análise-e-conclusão)
  - [V.1 Perfil de cada cluster](#V.1-Perfil-de-cada-cluster)
  - [V.2 Dando nome aos grupos](#V.2-Dando-nome-aos-grupos)
  - [V.3 Clusters × gêneros (com destaque para a música brasileira)](#V.3-Clusters-×-gêneros-(com-destaque-para-a-música-brasileira))
  - [V.4 Limitações do K-Means](#V.4-Limitações-do-K-Means)
  - [V.5 Próximos passos](#V.5-Próximos-passos)
- [Referências](#Referências)

> No Colab, o painel **Índice** (ícone ☰ à esquerda) também mostra esta estrutura.

## ⚙️ Configuração do ambiente

Esta célula prepara tudo o que o resto da aula usa. Ela fica aqui no começo, e não na Seção II,
porque a Seção I já desenha gráficos e roda um K-Means de exemplo. A Seção II explica cada
biblioteca com calma.

O que ela faz:

1. **Importa as bibliotecas**: `numpy` e `pandas` (dados), `matplotlib` e `seaborn` (gráficos) e
   as partes do `scikit-learn` usadas na aula (padronização, K-Means, métricas, PCA e dados de
   exemplo).
2. **Fixa a semente aleatória** `RANDOM_STATE = 42`. O K-Means e as amostragens sorteiam números;
   com a mesma semente, quem rodar a aula de novo vê **exatamente** os mesmos resultados.
3. **Define as constantes da análise**: o endereço do dataset (e o de uma cópia de reserva), as 9 *features* de áudio usadas
   no agrupamento, os gêneros que vamos destacar e o tamanho das amostras usadas nas partes mais
   caras (silhueta e gráficos de dispersão).
4. **Escolhe o estilo dos gráficos** e **imprime as versões** do Python e das bibliotecas, para
   que dê para comparar se algum resultado sair diferente.

In [1]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# Semente única de toda a aula (constituição IV): não crie outra.
RANDOM_STATE = 42

# Dataset público no Hugging Face (lido direto da internet, sem upload).
URL_DATASET = (
    "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset"
    "/resolve/main/dataset.csv"
)
# Cópia idêntica guardada no repositório do projeto, usada só se o Hugging Face falhar.
URL_DATASET_RESERVA = (
    "https://raw.githubusercontent.com/DataScience-Fatec-Projects/kmeans-musicas-spotify"
    "/main/data/spotify_tracks.csv"
)

# As 9 features contínuas de áudio que entram no K-Means (FR-035).
FEATURES = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
]

# Gêneros em destaque na Seção V (FR-034) e gêneros internacionais de contraste.
GENEROS_BR = ["brazil", "mpb", "sertanejo", "pagode", "samba", "forro", "funk"]
GENEROS_CONTRASTE = ["classical", "metal", "sleep", "hip-hop", "edm", "comedy"]

# Amostras para as etapas caras (research R3).
TAMANHO_AMOSTRA_SILHUETA = 10_000  # a silhueta exata é O(n²)
TAMANHO_AMOSTRA_GRAFICOS = 6_000   # dispersão legível, sem virar um borrão

# Estilo padrão dos gráficos.
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.figsize": (9, 5), "figure.dpi": 100, "axes.titleweight": "bold"})

print(f"Python       {sys.version.split()[0]}")
for nome, modulo in [("pandas", pd), ("numpy", np), ("scikit-learn", sklearn),
                     ("matplotlib", matplotlib), ("seaborn", sns)]:
    print(f"{nome:<12} {modulo.__version__}")

Python       3.12.13
pandas       3.1.1
numpy        2.5.5
scikit-learn 1.9.1
matplotlib   3.11.2
seaborn      0.13.2


Acima estão as versões usadas nesta execução. No Colab elas mudam de tempos em tempos; se algum
número da aula sair um pouco diferente do texto, compare primeiro estas versões.

## I. Introdução e Contextualização

### I.1 Aprendizado não supervisionado

> 🚧 **Em construção** — ver T021 ([#9](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/9)).

### I.2 Clustering (agrupamento)

> 🚧 **Em construção** — ver T021 ([#9](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/9)).

### I.3 Como o K-Means funciona — implementado do zero

> 🚧 **Em construção** — ver T022 ([#10](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/10)), T023 ([#11](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/11)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T024 ([#12](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/12)).

Parâmetro a mudar: `K_BRINQUEDO` e `SEMENTE_INICIAL`. O que observar: inicializações diferentes mudam as iterações e às vezes o resultado (mínimo local).

### I.4 Quanto custa? Complexidade do K-Means

> 🚧 **Em construção** — ver T025 ([#13](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/13)).

### I.5 Como avaliar: inércia, Método do Cotovelo e silhueta

> 🚧 **Em construção** — ver T026 ([#14](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/14)).

## II. Ferramentas de Trabalho (Scikit-learn e Python)

### II.1 Bibliotecas e importações

> 🚧 **Em construção** — ver T033 ([#16](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/16)).

### II.2 O KMeans do scikit-learn

> 🚧 **Em construção** — ver T034 ([#17](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/17)).

## III. Escolha do Tema e Preparação dos Dados

### III.1 Tema: por que agrupar músicas?

O Spotify tem dezenas de milhões de faixas. Ninguém consegue ouvir todas e anotar à mão se cada
uma é "calma", "dançante" ou "pesada". Mesmo assim, é exatamente esse tipo de informação que o
app usa o tempo todo:

- **Playlists automáticas**: "Para focar", "Treino", "Relaxar". Cada uma junta músicas que soam
  parecidas, mesmo sendo de artistas e gêneros diferentes.
- **Recomendação**: se você gosta de uma faixa, as faixas que estão no mesmo grupo dela são boas
  candidatas para tocar em seguida.
- **Descoberta de "climas"**: um samba e um pop podem ter a mesma energia dançante, e uma faixa
  de música clássica pode estar mais perto de um lo-fi do que de outra clássica. Os gêneros
  não contam essa história inteira.

Para cada faixa, o Spotify calculou números que descrevem o som: o quanto ela é dançante,
enérgica, acústica, alegre, falada etc. Não existe uma resposta certa dizendo a que grupo cada
música pertence. É um problema de **aprendizado não supervisionado** (Seção I): damos ao K-Means
só as características e deixamos que ele descubra os grupos.

> **A pergunta desta aula:** que perfis de som existem nas músicas do Spotify, e onde a música
> brasileira se encaixa neles?

O caminho até o modelo, nesta seção:

1. **III.2** conhecer o dataset e o que significa cada coluna;
2. **III.3** carregar e limpar os dados;
3. **III.4** escolher quais colunas entram no agrupamento;
4. **III.5** colocar todas as colunas na mesma escala.

### III.2 O dataset: origem e dicionário de variáveis

**Origem.** Usamos o [*Spotify Tracks Dataset*](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset), publicado no Hugging Face por
**Maharshi Pandya** sob a licença **BSD**. Os dados foram coletados com a API do Spotify e
trazem **114.000 linhas**: 1.000 faixas para cada um dos **114 gêneros**.

> ⚠️ Em novembro de 2024, o Spotify deixou de oferecer as *audio features* (dançabilidade,
> energia etc.) na sua API para novos aplicativos. Hoje não dá para coletar esses números de
> novo, e por isso trabalhamos com este retrato estático, lido direto do Hugging Face.

**Dicionário de variáveis.** Cada linha é uma combinação (faixa, gênero). As faixas de valores
abaixo são as dos dados brutos, antes da limpeza da III.3.

| Coluna | Significado | Valores | Entra no agrupamento? |
|---|---|---|---|
| `track_id` | identificador da faixa no Spotify | texto | não: é a chave |
| `artists` | artista(s), separados por `;` | texto | não: usado nos exemplos da V |
| `album_name` | nome do álbum | texto | não |
| `track_name` | nome da faixa | texto | não: usado nos exemplos da V |
| `popularity` | popularidade calculada pelo Spotify (quantidade e recência de execuções) | 0 a 100 | não: contexto, não é som |
| `duration_ms` | duração em milissegundos | 0 a ≈ 5,2 milhões (≈ 87 min) | não: contexto, não é som |
| `explicit` | a letra tem conteúdo explícito? | `True`/`False` | não: contexto, não é som |
| `danceability` | o quanto a faixa é boa para dançar (ritmo estável, batida forte) | 0 a 0,985 | **sim** |
| `energy` | intensidade percebida: rápida, alta, barulhenta | 0 a 1 | **sim** |
| `key` | tom da música (0 = Dó, 1 = Dó♯, …, 11 = Si) | 0 a 11 | não: categórica |
| `loudness` | volume médio, em decibéis (dB) | −49,5 a +4,5 | **sim** |
| `mode` | modo: 1 = maior, 0 = menor | 0 ou 1 | não: categórica |
| `speechiness` | presença de fala (perto de 1 = podcast, *stand-up*) | 0 a 0,965 | **sim** |
| `acousticness` | confiança de que a faixa é acústica | 0 a 0,996 | **sim** |
| `instrumentalness` | probabilidade de a faixa não ter voz | 0 a 1 | **sim** |
| `liveness` | probabilidade de ter sido gravada ao vivo, com plateia | 0 a 1 | **sim** |
| `valence` | positividade: alto = alegre, eufórica; baixo = triste, tensa | 0 a 0,995 | **sim** |
| `tempo` | andamento, em batidas por minuto (BPM) | 0 a 243 | **sim** (os zeros são inválidos, III.3) |
| `time_signature` | compasso estimado (3 = 3/4, 4 = 4/4…) | 0 a 5 | não: categórica |
| `track_genre` | gênero atribuído pela fonte | 114 gêneros | não: reservada para validar os grupos na V.3 |

A III.4 explica, uma por uma, por que essas 9 colunas entram e as outras ficam de fora.

### III.3 Carregamento e limpeza

**Carregando os dados.** O arquivo CSV (≈ 20 MB) é lido direto do endereço guardado em
`URL_DATASET`, sem upload e sem login, então o projeto roda igual em qualquer Colab.

Três detalhes do código:

- `index_col=0`: a primeira coluna do arquivo é só um número de linha, sem nome. Ela vira o
  índice da tabela e não uma coluna de dados.
- **Fonte de reserva**: se o Hugging Face não responder, o código tenta uma cópia idêntica do
  arquivo guardada no repositório do projeto, no GitHub (`URL_DATASET_RESERVA`). A saída diz de
  qual fonte os dados vieram.
- `try`/`except`: cada tentativa fica protegida. Se as duas fontes falharem, aparece uma mensagem
  em português com o link da página do dataset, em vez de só um erro técnico.

In [2]:
fontes = [("Hugging Face", URL_DATASET), ("cópia de reserva no GitHub", URL_DATASET_RESERVA)]

df_bruto = None
for nome_fonte, url in fontes:
    try:
        df_bruto = pd.read_csv(url, index_col=0)
        print(f"Dados carregados de: {nome_fonte}")
        break
    except Exception as erro:
        print(f"Não foi possível baixar de {nome_fonte}: {erro}")

if df_bruto is None:
    raise RuntimeError(
        "Nenhuma fonte do dataset respondeu. Confira a conexão com a internet ou baixe pela "
        "página original: https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset"
    )

linhas, colunas = df_bruto.shape
print(f"df_bruto: {linhas:,} linhas × {colunas} colunas".replace(",", "."))
df_bruto.head()

Dados carregados de: Hugging Face
df_bruto: 114.000 linhas × 20 colunas


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


A primeira linha da saída diz de qual fonte os dados vieram. Vieram **114.000 linhas e 20
colunas**, como a página do dataset descreve. As primeiras linhas
mostram as três partes da tabela: a identificação da faixa (`track_id`, artista, álbum, nome),
os números que descrevem o som (`danceability` até `tempo`) e, na última coluna, o gênero.

Agora vamos ver o **tipo** de cada coluna e se alguma tem valores faltando.

In [3]:
df_bruto.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  str    
 1   artists           113999 non-null  str    
 2   album_name        113999 non-null  str    
 3   track_name        113999 non-null  str    
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           114000 non-nu

Os tipos batem com o dicionário da III.2: as 9 *features* de áudio são números decimais
(`float64`), `key`, `mode` e `time_signature` são inteiros e `explicit` é verdadeiro/falso.

Repare na coluna **Non-Null Count**: `artists`, `album_name` e `track_name` têm **113.999**
valores, um a menos que as outras. Existe uma linha com dados faltando, e é por ela que a limpeza
começa.

**Limpando os dados.** Três regras, aplicadas nesta ordem:

1. **Linhas com valor ausente** são removidas. É a linha sem artista, álbum e nome que a
   `info()` mostrou. Sem nome, não daria para usá-la nos exemplos da Seção V.
2. **Linhas idênticas** (todas as 20 colunas iguais) são cópias acidentais e são removidas.
3. **`tempo` = 0** é removido. Nenhuma música tem 0 batidas por minuto: esse valor aparece em
   faixas de silêncio, ruído ou só fala, em que o Spotify não conseguiu medir o andamento. É um
   valor **inválido**, não um valor pequeno, e atrapalharia as distâncias.

O resultado se chama `df_pares`, porque cada linha ainda é um par (faixa, gênero).

In [4]:
sem_ausentes = df_bruto.dropna()
sem_identicas = sem_ausentes.drop_duplicates()
df_pares = sem_identicas[sem_identicas["tempo"] > 0].reset_index(drop=True)

limpeza = pd.DataFrame({
    "regra": ["1. valor ausente", "2. linha idêntica", "3. tempo = 0"],
    "linhas removidas": [
        len(df_bruto) - len(sem_ausentes),
        len(sem_ausentes) - len(sem_identicas),
        len(sem_identicas) - len(df_pares),
    ],
})
n_pares = f"{len(df_pares):,}".replace(",", ".")
print(f"df_pares: {n_pares} pares (faixa, gênero)")
limpeza

df_pares: 113.392 pares (faixa, gênero)


,regra,linhas removidas
0,1. valor ausente,1
1,2. linha idêntica,450
2,3. tempo = 0,157


A limpeza removeu **1 linha** com ausentes, **450 linhas idênticas** e **157 faixas com
`tempo` = 0**. Sobraram **113.392 pares**. Perdemos menos de 0,6% dos dados.

**Uma faixa, vários gêneros.** O dataset tem 1.000 linhas por gênero, e a mesma faixa pode
aparecer em mais de um: uma música pode estar em `mpb` e também em `brazil`. Quantos gêneros
cada faixa tem?

In [5]:
generos_por_faixa = df_pares.groupby("track_id")["track_genre"].nunique()
(generos_por_faixa.value_counts().sort_index()
    .rename_axis("gêneros da faixa").to_frame("faixas"))

,faixas
gêneros da faixa,
1,73284
2,11424
3,2955
4,1361
5,431
6,104
7,21
8,2
9,1


A maioria das faixas (73.284) aparece em um gênero só, mas **16.299 faixas aparecem em dois ou
mais**, e uma delas chega a 9 gêneros. Juntas, essas repetições somam 23.809 linhas a mais.

**Por que isso importa para o K-Means?** O centroide de um grupo é a **média** dos pontos do
grupo. Se treinarmos com `df_pares`, uma faixa que está em 9 gêneros entra 9 vezes nessa média e
puxa o centroide para perto dela, só porque a fonte a classificou em mais gêneros. Nenhuma faixa
deveria pesar mais que outra.

Por isso a base de treino terá **uma linha por faixa**. Antes, conferimos com um `assert` (uma
checagem que interrompe o notebook se falhar) que as cópias de uma faixa têm exatamente os
mesmos valores de áudio, para garantir que não perdemos informação ao ficar com uma delas.

Repare que **não escolhemos nem descartamos nenhum gênero**: o treino nem olha para o gênero, e na
Seção V o grupo de cada faixa volta para todos os gêneros em que ela aparece.

In [6]:
faixas_divergentes = (df_pares.groupby("track_id")[FEATURES].nunique() > 1).any(axis=1).sum()
assert faixas_divergentes == 0, f"{faixas_divergentes} faixas têm áudio diferente entre as cópias"

df_faixas = (df_pares.drop_duplicates("track_id")
             .drop(columns="track_genre")
             .reset_index(drop=True))
print(f"Faixas com áudio diferente entre as cópias: {faixas_divergentes}")
n_faixas = f"{len(df_faixas):,}".replace(",", ".")
n_pares = f"{len(df_pares):,}".replace(",", ".")
print(f"df_faixas: {n_faixas} faixas únicas (base de treino)")
print(f"df_pares:  {n_pares} pares, com {df_pares['track_genre'].nunique()} gêneros "
      "(guardado para a análise por gênero da V.3)")

Faixas com áudio diferente entre as cópias: 0
df_faixas: 89.583 faixas únicas (base de treino)
df_pares:  113.392 pares, com 114 gêneros (guardado para a análise por gênero da V.3)


O `assert` passou: **nenhuma** faixa tem valores de áudio diferentes entre as cópias. A base de
treino `df_faixas` tem **89.583 faixas únicas**.

Duas decisões ficam registradas aqui:

- `df_faixas` **não tem a coluna `track_genre`**. Ela foi retirada de propósito: como cada faixa
  pode ter vários gêneros, guardar só um seria arbitrário, e o gênero não deve chegar perto do
  treino (III.4).
- `df_pares` continua guardado, com os 114 gêneros. Na Seção V, cada faixa recebe o seu cluster,
  e esse rótulo volta para **todas** as linhas dela em `df_pares`. Assim nenhum gênero perde
  faixas na análise.

### III.4 Escolha das features

Uma *feature* (característica) é uma coluna que o K-Means usa para medir se duas músicas são
parecidas. A pergunta desta aula é sobre **som**, então entram só as colunas que descrevem o som,
e todas elas precisam ser números contínuos, para que a distância entre duas faixas faça sentido.

| Grupo | Colunas | Decisão e motivo |
|---|---|---|
| Som, contínuas | `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo` | **entram** (são as 9 `FEATURES`) |
| Som, categóricas | `key`, `mode`, `time_signature` | **ficam fora**: são códigos, não quantidades. O tom 11 (Si) não é "o dobro" do tom 5 (Fá), e a distância entre os números não significa nada musicalmente |
| Contexto | `popularity`, `duration_ms`, `explicit` | **ficam fora**: dizem como a faixa foi recebida ou embalada, não como ela soa. Elas voltam na Seção V para ajudar a interpretar os grupos |
| Identificação | `track_id`, `artists`, `album_name`, `track_name` | **ficam fora**: são nomes, não medidas. Servem para os exemplos da Seção V |
| Rótulo da fonte | `track_genre` | **fica fora, de propósito**: é a "resposta" que vamos usar na V.3 para conferir se os grupos fazem sentido. Se ela entrasse no treino, o K-Means apenas copiaria os gêneros e a conferência viraria um círculo vicioso (o chamado **vazamento de rótulo**) |

Vamos olhar o resumo estatístico das 9 features escolhidas:

In [7]:
df_faixas[FEATURES].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
danceability,89583.0,0.563,0.175,0.051,0.450,0.576,0.692,0.985
energy,89583.0,0.635,0.256,0.000,0.458,0.677,0.853,1.000
loudness,89583.0,-8.475,5.179,-46.591,-10.308,-7.177,-5.104,4.532
speechiness,89583.0,0.088,0.113,0.022,0.036,0.049,0.086,0.965
acousticness,89583.0,0.328,0.338,0.000,0.017,0.188,0.624,0.996
instrumentalness,89583.0,0.173,0.323,0.000,0.000,0.000,0.094,1.000
liveness,89583.0,0.217,0.194,0.009,0.098,0.132,0.278,1.000
valence,89583.0,0.470,0.262,0.000,0.250,0.458,0.683,0.995
tempo,89583.0,122.272,29.707,30.200,99.515,122.021,140.087,243.372


A tabela confirma que não há valores faltando (`count` = 89.583 em todas) e revela o problema que
a III.5 resolve: **as escalas são muito diferentes**.

- Sete features vão de **0 a 1** (`danceability`, `energy`, `speechiness`, `acousticness`,
  `instrumentalness`, `liveness`, `valence`), com desvio-padrão (`std`) entre 0,11 e 0,34.
- `loudness` vai de **−46,6 a +4,5 dB**, com desvio de 5,2.
- `tempo` vai de **30 a 243 BPM**, com desvio de 29,7, cerca de **170 vezes** o desvio de
  `danceability` (0,175).

Se usarmos esses números como estão, uma diferença de 30 BPM vai "valer" muito mais do que a
diferença entre uma música nada dançante (0) e uma totalmente dançante (1).

### III.5 Padronização com StandardScaler

**Por que o K-Means precisa disso?** O K-Means agrupa pela **distância euclidiana**, a mesma
"distância em linha reta" da geometria: ele soma as diferenças ao quadrado de todas as features.
Uma feature com números grandes produz diferenças grandes e domina a soma. O agrupamento passaria
a ser praticamente sobre `tempo` e `loudness`, ignorando as outras sete.

A **padronização** resolve isso transformando cada feature com a fórmula

$$z = \frac{x - \text{média}}{\text{desvio-padrão}}$$

Depois dela, toda feature tem **média 0 e desvio-padrão 1**. Cada valor passa a dizer "quantos
desvios acima ou abaixo da média" a faixa está, e uma diferença de 1 tem o mesmo peso em
qualquer feature.

No código, o `StandardScaler` do scikit-learn **aprende** a média e o desvio de cada feature com
`fit` e **aplica** a fórmula com `transform` (`fit_transform` faz os dois). O `scaler` fica
guardado: na Seção V ele faz a conta inversa (`inverse_transform`), para descrever os grupos de
volta em BPM, dB e assim por diante.

In [8]:
scaler = StandardScaler()
X = scaler.fit_transform(df_faixas[FEATURES])

escalas = pd.DataFrame({
    "média antes": df_faixas[FEATURES].mean(),
    "desvio antes": df_faixas[FEATURES].std(ddof=0),
    "média depois": X.mean(axis=0),
    "desvio depois": X.std(axis=0),
}, index=FEATURES)
print(f"X: {X.shape[0]:,} faixas × {X.shape[1]} features".replace(",", "."))
escalas.round(3) + 0.0  # o "+ 0.0" só troca o -0.0 por 0.0 na exibição

X: 89.583 faixas × 9 features


,média antes,desvio antes,média depois,desvio depois
danceability,0.563,0.175,0.0,1.0
energy,0.635,0.256,0.0,1.0
loudness,-8.475,5.179,0.0,1.0
speechiness,0.088,0.113,0.0,1.0
acousticness,0.328,0.338,0.0,1.0
instrumentalness,0.173,0.323,0.0,1.0
liveness,0.217,0.194,0.0,1.0
valence,0.470,0.262,0.0,1.0
tempo,122.272,29.707,0.0,1.0


Antes, as médias iam de 0,09 (`speechiness`) a 122 (`tempo`), e os desvios de 0,11 a 29,7.
Depois da padronização, **todas as 9 colunas de `X` têm média 0 e desvio 1**. `X` é a matriz que
o K-Means vai receber na Seção IV: 89.583 linhas (faixas) e 9 colunas (features).

**Um exemplo concreto.** Vamos comparar as duas primeiras faixas da base e ver quanto cada
feature contribui para a distância entre elas, sem e com padronização.

In [9]:
faixa_a, faixa_b = df_faixas.iloc[0], df_faixas.iloc[1]
print(f"A: {faixa_a['track_name']} ({faixa_a['artists']})")
print(f"B: {faixa_b['track_name']} ({faixa_b['artists']})")

dif2_bruta = (faixa_a[FEATURES].astype(float) - faixa_b[FEATURES].astype(float)) ** 2
dif2_padronizada = pd.Series((X[0] - X[1]) ** 2, index=FEATURES)

contribuicao = pd.DataFrame({
    "sem padronizar (%)": 100 * dif2_bruta / dif2_bruta.sum(),
    "padronizado (%)": 100 * dif2_padronizada / dif2_padronizada.sum(),
}).round(1)
print(f"Distância sem padronizar: {np.sqrt(dif2_bruta.sum()):.2f}")
print(f"Distância padronizada:    {np.sqrt(dif2_padronizada.sum()):.2f}")
contribuicao

A: Comedy (Gen Hoshino)
B: Ghost - Acoustic (Ben Woodward)
Distância sem padronizar: 14.83
Distância padronizada:    4.43


,sem padronizar (%),padronizado (%)
danceability,0.0,10.9
energy,0.0,6.8
loudness,50.0,20.9
speechiness,0.0,1.8
acousticness,0.4,35.4
instrumentalness,0.0,0.0
liveness,0.0,8.9
valence,0.1,14.8
tempo,49.4,0.6


Sem padronizar, **`loudness` (50,0%) e `tempo` (49,4%) respondem por 99,4% da distância** entre
as duas faixas, e `danceability`, que é bem diferente entre elas (0,68 contra 0,42), contribui
com **0,0%**. Para o K-Means, essas faixas seriam diferentes só porque uma é mais alta e mais
rápida que a outra.

Depois de padronizar, a distância passa a refletir o som como um todo: a maior contribuição vem
de `acousticness` (35,4%), já que "Ghost - Acoustic" é acústica e a outra não. `tempo`, cuja
diferença de 10 BPM é pequena para os padrões do dataset, cai para 0,6%. É por isso que o
K-Means sempre vem depois da padronização.

> **Resumo da Seção III:** `df_faixas` (89.583 faixas únicas), `df_pares` (113.392 pares faixa ×
> gênero, para a Seção V), `FEATURES` (9 colunas de som), `scaler` (para voltar à escala
> original) e `X` (a matriz padronizada que entra no K-Means).

## IV. Implementação e Aplicação Prática

### IV.1 Definindo k: Método do Cotovelo (e silhueta)

O K-Means precisa que a gente diga **quantos grupos (k)** procurar. Não existe um k "certo"
escrito nos dados, então vamos decidir com três critérios, nesta ordem:

1. **Método do Cotovelo**: onde a inércia para de cair rápido;
2. **Silhueta**: entre os candidatos do cotovelo, qual separa melhor os grupos;
3. **Interpretabilidade**: se cada grupo tem um perfil musical que dá para descrever.

**Inércia** é a soma das distâncias ao quadrado de cada faixa até o centroide do seu grupo:
quanto menor, mais "apertados" estão os grupos. Ela **sempre** diminui quando k aumenta (com mais
centroides, cada ponto fica mais perto de algum), então não adianta procurar o k com a menor
inércia. O que interessa é **onde a queda desacelera**: a partir dali, criar mais grupos quase não
melhora nada. No gráfico, esse ponto parece o cotovelo de um braço dobrado.

O código treina um K-Means para cada k de 1 a 10 com os dados padronizados `X`. Os modelos ficam
guardados em `modelos_por_k` para serem reaproveitados pela silhueta e pela análise dos perfis,
sem treinar tudo de novo. `n_init=10` roda o K-Means com 10 inicializações diferentes e fica com a
melhor, para não depender da sorte do sorteio inicial.

In [ ]:
valores_k = range(1, 11)
modelos_por_k = {
    k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X) for k in valores_k
}
inercias = [modelos_por_k[k].inertia_ for k in valores_k]

cotovelo = pd.DataFrame({"inércia": inercias}, index=pd.Index(valores_k, name="k"))
cotovelo["queda em relação ao k anterior (%)"] = (-100 * cotovelo["inércia"].pct_change()).round(1)

fig, ax = plt.subplots()
ax.plot(list(valores_k), inercias, marker="o", label="inércia")
ax.set(title="Método do Cotovelo: inércia para cada número de clusters",
       xlabel="número de clusters (k)", ylabel="inércia (soma das distâncias²)")
ax.set_xticks(list(valores_k))
ax.legend()
plt.show()
cotovelo.assign(inércia=cotovelo["inércia"].round().astype(int))

A curva é **suave, sem um cotovelo nítido**, e isso é comum em dados reais. A tabela ajuda a ver
onde a queda desacelera:

- de k = 1 para k = 2 a inércia cai **21,5%**: é a primeira dobra;
- de k = 2 até k = 7, cada grupo a mais ainda remove **9 a 11%** da inércia (11,3%, 10,0%, 9,7%,
  9,2%, 9,1%);
- depois de k = 7, a queda cai para **5,6%**, 5,2% e 4,1%: é a segunda dobra, mais suave.

O cotovelo sozinho não decide. Ele indica que vale a pena ir além de k = 2 e que, depois de k = 7,
cada grupo a mais rende pouco. Os candidatos ficam entre **k = 5 e k = 7**. A silhueta ajuda a
escolher entre eles.

**Silhueta.** Para cada faixa, a silhueta compara duas distâncias: a média até as faixas do
**próprio** grupo e a média até as faixas do grupo **vizinho mais próximo**. O resultado vai de
−1 a 1:

- perto de **1**: a faixa está bem no meio do seu grupo, longe dos outros;
- perto de **0**: a faixa está na fronteira entre dois grupos;
- **negativo**: a faixa parece estar no grupo errado.

A nota do agrupamento é a média das silhuetas de todas as faixas. Diferente da inércia, ela **não**
melhora sozinha quando k aumenta, então dá para comparar valores de k diferentes. A silhueta só
existe a partir de k = 2.

**Por que usar uma amostra?** A silhueta exata compara cada faixa com todas as outras: são
n² distâncias, ou seja, complexidade **O(n²)**. Com 89.583 faixas, seriam cerca de 8 bilhões de
distâncias, o que levaria minutos e muitos GB de memória. Com uma amostra de
`TAMANHO_AMOSTRA_SILHUETA = 10.000` faixas (≈ 100 milhões de distâncias), cada k leva cerca de
1 segundo. Como a amostra é sorteada com `RANDOM_STATE`, é sempre a mesma, e todos os k são
avaliados com as mesmas faixas.

In [ ]:
silhuetas = [
    silhouette_score(X, modelos_por_k[k].labels_,
                     sample_size=TAMANHO_AMOSTRA_SILHUETA, random_state=RANDOM_STATE)
    for k in range(2, 11)
]
tabela_silhueta = pd.DataFrame({"silhueta": silhuetas}, index=pd.Index(range(2, 11), name="k"))
melhor_k = tabela_silhueta.loc[3:, "silhueta"].idxmax()

fig, ax = plt.subplots()
ax.plot(tabela_silhueta.index, tabela_silhueta["silhueta"], marker="o", label="silhueta média")
ax.scatter([melhor_k], [tabela_silhueta.loc[melhor_k, "silhueta"]], s=200, facecolors="none",
           edgecolors="red", linewidths=2, label=f"maior silhueta com k ≥ 3 (k = {melhor_k})")
ax.set(title=f"Silhueta para cada número de clusters (amostra de {TAMANHO_AMOSTRA_SILHUETA:,} faixas)".replace(",", "."),
       xlabel="número de clusters (k)", ylabel="silhueta média (−1 a 1)")
ax.set_xticks(range(2, 11))
ax.legend()
plt.show()
tabela_silhueta.round(3)

- **k = 2** tem a maior silhueta (**0,260**), mas dois grupos são genéricos demais para responder
  à pergunta da aula ("que perfis de som existem?"). Por isso a comparação fica entre os
  candidatos que o cotovelo indicou.
- Entre os candidatos, a silhueta cresce de **0,177 (k = 5)** para **0,189 (k = 6)** e chega ao
  máximo em **0,196 (k = 7)**, a maior de todas para k ≥ 3. Depois de k = 7 ela volta a cair
  (0,177 em k = 8).
- Todos os valores são **baixos** (abaixo de 0,2). Isso quer dizer que os grupos de músicas se
  **sobrepõem**: muitas faixas ficam na fronteira entre dois perfis. Faz sentido, porque não existe
  uma linha nítida entre "dançante" e "intenso". Voltamos a isso nas limitações (V.4).

Cotovelo e silhueta apontam para o mesmo lugar: **k = 7**. Falta o terceiro critério.

**Interpretabilidade.** Um k só é útil se cada grupo puder ser descrito em palavras. Para comparar
os candidatos, a tabela abaixo mostra, para cada grupo de k = 5, 6 e 7, o tamanho e as **duas
features mais marcantes** do seu centroide. Os valores estão em **desvios-padrão em relação à
média**, por causa da padronização: +1,9 em `instrumentalness` quer dizer "quase dois desvios acima
da média das faixas".

In [ ]:
candidatos = [5, 6, 7]
resumo_candidatos = pd.DataFrame({
    "inércia": [inercias[k - 1] for k in candidatos],
    "silhueta": [silhuetas[k - 2] for k in candidatos],
}, index=pd.Index(candidatos, name="k")).round({"inércia": 0, "silhueta": 3})
display(resumo_candidatos)

linhas = []
for k in candidatos:
    centros = pd.DataFrame(modelos_por_k[k].cluster_centers_, columns=FEATURES)
    tamanhos_k = np.bincount(modelos_por_k[k].labels_)
    for c in range(k):
        marcantes = centros.loc[c].abs().nlargest(2).index
        linhas.append({
            "k": k, "cluster": c, "faixas": tamanhos_k[c],
            "features mais marcantes (desvios da média)":
                ", ".join(f"{f} {centros.loc[c, f]:+.1f}" for f in marcantes),
        })
pd.DataFrame(linhas).set_index(["k", "cluster"])

Lendo os perfis:

- **k = 5** já mostra cinco perfis claros: alegre e dançante (`valence` +0,8, `danceability`
  +0,7), acústico e calmo (`acousticness` +1,0, `energy` −1,0), intenso (`energy` +0,7), fala
  (`speechiness` +6,5) e ambiente/instrumental (`loudness` −2,3, `instrumentalness` +1,9).
- **k = 6** acrescenta um grupo de **6.630 faixas gravadas ao vivo** (`liveness` +2,6).
- **k = 7** mantém esses seis e separa ainda um grupo de **10.094 faixas instrumentais e
  enérgicas** (`instrumentalness` +1,9, `acousticness` −0,6), que no k = 5 estava misturado com
  os outros.

Nenhum grupo de k = 7 é um "resto" sem sentido: todos têm pelo menos uma feature a mais de 0,7
desvio da média.

**Decisão: k = 7.** Os três critérios concordam:

| Critério | k = 5 | k = 6 | **k = 7** |
|---|---|---|---|
| 1. Cotovelo: queda da inércia no passo seguinte | 9,2% | 9,1% | **5,6%** (é onde desacelera) |
| 2. Silhueta | 0,177 | 0,189 | **0,196** (a maior com k ≥ 3) |
| 3. Perfis interpretáveis | 5 | 6 | **7** |

O k = 5 foi **descartado** porque deixa de lado dois perfis reais (ao vivo e eletrônica
instrumental) e tem a menor silhueta dos três. O k = 6 foi **descartado** porque ainda mistura a
eletrônica instrumental com outros grupos e o cotovelo mostra que o passo seguinte, de 6 para 7,
ainda remove 9,1% da inércia. Com k = 8, a silhueta cai para 0,177 e a inércia só cai mais 5,6%.

In [ ]:
K_ESCOLHIDO = 7

`K_ESCOLHIDO` é a constante que o resto da aula usa. Quem quiser ver como ficariam os grupos com
outro k pode usar a célula **🧪 Experimente** da IV.2, sem mudar esta decisão.

### IV.2 Treinando o K-Means

> 🚧 **Em construção** — ver T016 ([#29](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/29)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T017 ([#30](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/30)).

Parâmetro a mudar: `K_TESTE`. O que observar: como os perfis se dividem ou se fundem com mais ou menos clusters.

### IV.3 Visualizando a separação dos clusters

> 🚧 **Em construção** — ver T018 ([#31](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/31)).

### IV.4 Centroides em relação aos pontos

> 🚧 **Em construção** — ver T019 ([#32](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/32)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T020 ([#33](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/33)).

Parâmetro a mudar: `USAR_PADRONIZACAO = True/False`. O que observar: sem padronizar, `tempo` e `loudness` dominam a distância e os clusters mudam.

## V. Análise e conclusão

### V.1 Perfil de cada cluster

> 🚧 **Em construção** — ver T027 ([#35](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/35)).

### V.2 Dando nome aos grupos

> 🚧 **Em construção** — ver T028 ([#36](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/36)).

### V.3 Clusters × gêneros (com destaque para a música brasileira)

> 🚧 **Em construção** — ver T029 ([#37](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/37)), T030 ([#38](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/38)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T031 ([#39](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/39)).

Parâmetro a mudar: `GENEROS_EXTRA`. O que observar: onde caem os gêneros que você escolher.

### V.4 Limitações do K-Means

> 🚧 **Em construção** — ver T032 ([#40](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/40)).

### V.5 Próximos passos

> 🚧 **Em construção** — ver T032 ([#40](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/40)).

## Referências

> 🚧 **Em construção** — ver T035 ([#42](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/42)).